In [1]:
import pandas as pd
import numpy as np
accounts = pd.read_csv(r"D:\saas360\ravenstack_accounts.csv")
subscriptions = pd.read_csv(r"D:\saas360\ravenstack_subscriptions.csv")
feature_usage = pd.read_csv(r"D:\saas360\ravenstack_feature_usage.csv")
support_tickets = pd.read_csv(r"D:saas360\ravenstack_support_tickets.csv")
churn_events = pd.read_csv(r"D:saas360\ravenstack_churn_events.csv")

In [2]:
accounts_clean = accounts.copy()
subscriptions_clean = subscriptions.copy()
feature_usage_clean = feature_usage.copy()
support_tickets_clean = support_tickets.copy()
churn_events_clean = churn_events.copy()

In [3]:
def clean_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df
    

In [4]:
accounts_clean = clean_column_names(accounts_clean)
subscriptions_clean = clean_column_names(subscriptions_clean)
feature_usage_clean = clean_column_names(feature_usage_clean)
support_tickets_clean = clean_column_names(support_tickets_clean)
churn_events_clean = clean_column_names(churn_events_clean)

In [5]:
accounts_clean.columns


Index(['account_id', 'account_name', 'industry', 'country', 'signup_date',
       'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag'],
      dtype='object')

In [7]:
text_columns = [
    "account_name",
    "industry",
    "country",
    "referral_source",
    "plan_tier"
]

for col in text_columns:
    accounts_clean[col] = accounts_clean[col].str.strip()

In [8]:
subscriptions_clean["plan_tier"] = (
    subscriptions_clean["plan_tier"].str.strip()
)

subscriptions_clean["billing_frequency"] = (
    subscriptions_clean["billing_frequency"].str.strip()
)

feature_usage_clean["feature_name"] = (
    feature_usage_clean["feature_name"].str.strip()
)

support_tickets_clean["priority"] = (
    support_tickets_clean["priority"].str.strip()
)

churn_events_clean["reason_code"] = (
    churn_events_clean["reason_code"].str.strip()
)

In [9]:
accounts_clean["signup_date"] = pd.to_datetime(
    accounts_clean["signup_date"],
    errors="coerce"
)
subscriptions_clean["start_date"] = pd.to_datetime(
    subscriptions_clean["start_date"],
    errors="coerce"
)

subscriptions_clean["end_date"] = pd.to_datetime(
    subscriptions_clean["end_date"],
    errors="coerce"
)
feature_usage_clean["usage_date"] = pd.to_datetime(
    feature_usage_clean["usage_date"],
    errors="coerce"
)
support_tickets_clean["submitted_at"] = pd.to_datetime(
    support_tickets_clean["submitted_at"],
    errors="coerce"
)

support_tickets_clean["closed_at"] = pd.to_datetime(
    support_tickets_clean["closed_at"],
    errors="coerce"
)
churn_events_clean["churn_date"] = pd.to_datetime(
    churn_events_clean["churn_date"],
    errors="coerce"
)

In [10]:
subscriptions_clean[
    ["seats", "mrr_amount", "arr_amount"]
].dtypes

seats         int64
mrr_amount    int64
arr_amount    int64
dtype: object

In [11]:
subscriptions_clean["seats"] = pd.to_numeric(
    subscriptions_clean["seats"],
    errors="coerce"
)

subscriptions_clean["mrr_amount"] = pd.to_numeric(
    subscriptions_clean["mrr_amount"],
    errors="coerce"
)

subscriptions_clean["arr_amount"] = pd.to_numeric(
    subscriptions_clean["arr_amount"],
    errors="coerce"
)

feature_usage_clean["usage_count"] = pd.to_numeric(
    feature_usage_clean["usage_count"],
    errors="coerce"
)

feature_usage_clean["usage_duration_secs"] = pd.to_numeric(
    feature_usage_clean["usage_duration_secs"],
    errors="coerce"
)

feature_usage_clean["error_count"] = pd.to_numeric(
    feature_usage_clean["error_count"],
    errors="coerce"
)

In [12]:
duplicate_usage_ids = feature_usage_clean[
    feature_usage_clean["usage_id"].duplicated(keep=False)
].sort_values("usage_id")

duplicate_usage_ids

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False
22,U-25b56c,S-810c27,2024-10-06,feature_20,7,231,0,False
7574,U-25b56c,S-34253c,2023-10-28,feature_20,6,2166,1,False
21376,U-48a4aa,S-93f835,2024-08-24,feature_39,10,4770,0,False
1085,U-48a4aa,S-383ac2,2023-02-12,feature_28,14,7126,0,False


In [15]:
duplicate_usage_ids.groupby("usage_id").size()


usage_id
U-0c9318    2
U-13ce5b    2
U-2103bb    2
U-25b56c    2
U-48a4aa    2
U-4ef9da    2
U-52f4ae    2
U-5cd3c2    2
U-649cec    2
U-662254    2
U-6970d1    2
U-74b23f    2
U-7a87ed    2
U-7c6703    2
U-8128da    2
U-8a3738    2
U-ae1a6c    2
U-b0ae95    2
U-c878da    2
U-d20c6d    2
U-f42b0a    2
dtype: int64

In [16]:
duplicate_usage_ids

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False
22,U-25b56c,S-810c27,2024-10-06,feature_20,7,231,0,False
7574,U-25b56c,S-34253c,2023-10-28,feature_20,6,2166,1,False
21376,U-48a4aa,S-93f835,2024-08-24,feature_39,10,4770,0,False
1085,U-48a4aa,S-383ac2,2023-02-12,feature_28,14,7126,0,False


In [17]:
feature_usage_clean = feature_usage_clean.drop_duplicates()

In [18]:
feature_usage_clean["usage_event_key"] = (
    feature_usage_clean["usage_id"].astype(str)
    + "_"
    + feature_usage_clean.groupby("usage_id").cumcount().astype(str)
)

In [19]:
subscriptions_clean["end_date"].isna().sum()

4514

In [20]:
subscriptions_clean["subscription_status"] = np.where(
    subscriptions_clean["end_date"].isna(),
    "Active",
    "Ended"
)

In [21]:
support_tickets_clean["survey_responded"] = (
    support_tickets_clean["satisfaction_score"].notna()
)

In [22]:
churn_events_clean["feedback_text"]

0      switched to competitor
1                         NaN
2            missing features
3      switched to competitor
4               too expensive
                ...          
595    switched to competitor
596                       NaN
597             too expensive
598                       NaN
599                       NaN
Name: feedback_text, Length: 600, dtype: object

In [23]:
usage_check = feature_usage_clean.merge(
    subscriptions_clean[
        ["subscription_id", "start_date", "end_date"]
    ],
    on="subscription_id",
    how="left"
)

In [24]:
usage_check["usage_timing_status"] = np.select(
    [
        usage_check["usage_date"] < usage_check["start_date"],
        
        usage_check["end_date"].notna() &
        (usage_check["usage_date"] > usage_check["end_date"])
    ],
    [
        "Before subscription",
        "After subscription"
    ],
    default="Within subscription"
)

In [25]:
usage_check["usage_timing_status"].value_counts()

usage_timing_status
Before subscription    19142
Within subscription     5568
After subscription       290
Name: count, dtype: int64

In [26]:
feature_usage_valid = usage_check[
    (
        usage_check["usage_date"] >=
        usage_check["start_date"]
    )
    &
    (
        usage_check["end_date"].isna()
        |
        (usage_check["usage_date"] <= usage_check["end_date"])
    )
].copy()

In [27]:
revenue_check = subscriptions_clean[
    ~np.isclose(
        subscriptions_clean["arr_amount"],
        subscriptions_clean["mrr_amount"] * 12
    )
]

print("Revenue mismatches:", len(revenue_check))

Revenue mismatches: 0


In [28]:
print(
    "Duplicate account IDs:",
    accounts_clean["account_id"].duplicated().sum()
)

print(
    "Duplicate subscription IDs:",
    subscriptions_clean["subscription_id"].duplicated().sum()
)

print(
    "Duplicate ticket IDs:",
    support_tickets_clean["ticket_id"].duplicated().sum()
)

print(
    "Duplicate churn IDs:",
    churn_events_clean["churn_event_id"].duplicated().sum()
)

Duplicate account IDs: 0
Duplicate subscription IDs: 0
Duplicate ticket IDs: 0
Duplicate churn IDs: 0


In [30]:
import os

os.makedirs(r"D:\saas360\cleaned", exist_ok=True)

print("Cleaned data folder is ready.")

Cleaned data folder is ready.


In [31]:
accounts_clean.to_csv(
    r"D:\saas360\cleaned\accounts_clean.csv",
    index=False
)

subscriptions_clean.to_csv(
    r"D:\saas360\cleaned\subscriptions_clean.csv",
    index=False
)

feature_usage_clean.to_csv(
    r"D:\saas360\cleaned\feature_usage_clean.csv",
    index=False
)

support_tickets_clean.to_csv(
    r"D:\saas360\cleaned\support_tickets_clean.csv",
    index=False
)

churn_events_clean.to_csv(
    r"D:\saas360\cleaned\churn_events_clean.csv",
    index=False
)

feature_usage_valid.to_csv(
    r"D:\saas360\cleaned\feature_usage_valid.csv",
    index=False
)

print("All cleaned files saved successfully!")

All cleaned files saved successfully!


In [32]:
os.listdir(r"D:\saas360\cleaned")

['accounts_clean.csv',
 'churn_events_clean.csv',
 'feature_usage_clean.csv',
 'feature_usage_valid.csv',
 'subscriptions_clean.csv',
 'support_tickets_clean.csv']